In [ ]:
import numpy as np

import numpy.linalg as la

import matplotlib.pyplot as plt

import cv2

from PIL import Image

import pickle

import os

In [ ]:
DATA_DIR = './cifar-10-batches-py'

# Load a single CIFAR-10 batch from disk
def load_batch(filepath):

    with open(filepath, 'rb') as f:

        d = pickle.load(f, encoding='bytes')

    images = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

    labels = np.array(d[b'labels'])

    return images, labels

# Load human-readable class names from batches.meta
def load_class_names(meta_filepath):

    with open(meta_filepath, 'rb') as f:

        meta = pickle.load(f, encoding='bytes')

    class_names = [name.decode('utf-8') for name in meta[b'label_names']]

    return class_names

# Load all 5 training batches and concatenate
train_batches = [load_batch(os.path.join(DATA_DIR, f'data_batch_{i}')) for i in range(1, 6)]

train_images = np.concatenate([b[0] for b in train_batches])

train_labels = np.concatenate([b[1] for b in train_batches])

# Load test batch
test_images, test_labels = load_batch(os.path.join(DATA_DIR, 'test_batch'))

# Load class names
class_names = load_class_names(os.path.join(DATA_DIR, 'batches.meta'))

print(len(train_images), len(test_images), train_images[0].shape, class_names)

In [ ]:
# Extract a 256-dim normalized LBP histogram from an RGB image
def extract_lbp(img_rgb):

    # Convert to grayscale
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    # 8-neighbor offsets in clockwise order starting top-left
    offsets = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]

    h, w = gray.shape

    center = gray[1:-1, 1:-1].astype(np.float32)

    lbp = np.zeros((h - 2, w - 2), dtype=np.uint8)

    # Vectorized: compare each neighbor slice to center slice
    for k_bit, (di, dj) in enumerate(offsets):

        neighbor = gray[1 + di:h - 1 + di, 1 + dj:w - 1 + dj].astype(np.float32)

        lbp |= ((neighbor >= center).astype(np.uint8) << k_bit)

    hist, _ = np.histogram(lbp.ravel(), bins=256, range=(0, 256))

    feat = hist.astype(np.float32)

    feat = feat / (feat.sum() + 1e-7)

    return feat

feat_test = extract_lbp(train_images[0])

assert feat_test.shape == (256,)

assert feat_test.dtype == np.float32

In [ ]:
k = 10

N_QUERIES = 100

TOTAL_RELEVANT = 5000

db_features_lbp = np.zeros((len(train_images), 256), dtype=np.float32)

# Compute LBP feature for every training image
for i in range(len(train_images)):

    db_features_lbp[i] = extract_lbp(train_images[i])

    if i % 10000 == 0:

        print(f'Processed {i}/50000')

assert db_features_lbp.shape == (50000, 256)

print(db_features_lbp.shape, db_features_lbp.dtype)

In [ ]:
# Retrieve top-k nearest neighbours by L2 distance
def retrieve(query_vec, db_features, k=10):

    dists = la.norm(db_features - query_vec, axis=1)

    top_k_idx = np.argsort(dists)[:k]

    return top_k_idx, dists[top_k_idx]

# Smoke test retrieval
query_idx = 0

query_vec = db_features_lbp[0]

top_k_idx, top_k_dists = retrieve(query_vec, db_features_lbp, k=10)

assert len(top_k_idx) == 10

assert top_k_dists[0] <= top_k_dists[-1]

print(top_k_idx, top_k_dists)

In [ ]:
# Fraction of retrieved results matching the query label
def precision_at_k(query_label, retrieved_labels, k):

    count = np.sum(retrieved_labels[:k] == query_label)

    return count / k

# Fraction of all relevant items retrieved in top-k
def recall_at_k(query_label, retrieved_labels, k, total_relevant):

    count = np.sum(retrieved_labels[:k] == query_label)

    return count / total_relevant

# Reciprocal rank of the first correct result
def reciprocal_rank(query_label, retrieved_labels):

    for i, label in enumerate(retrieved_labels):

        if label == query_label:

            return 1.0 / (i + 1)

    return 0.0

np.random.seed(42)

query_indices = np.random.choice(len(test_images), size=N_QUERIES, replace=False)

precisions = []

recalls = []

rr_scores = []

for qi in query_indices:

    q_img = test_images[qi]

    q_label = test_labels[qi]

    q_feat = extract_lbp(q_img)

    top_k_idx, _ = retrieve(q_feat, db_features_lbp, k=k)

    retrieved_labels = train_labels[top_k_idx]

    precisions.append(precision_at_k(q_label, retrieved_labels, k))

    recalls.append(recall_at_k(q_label, retrieved_labels, k, TOTAL_RELEVANT))

    rr_scores.append(reciprocal_rank(q_label, retrieved_labels))

mean_precision = np.mean(precisions)

mean_recall = np.mean(recalls)

mrr = np.mean(rr_scores)

print(f'LBP | P@{k}: {mean_precision:.3f} | R@{k}: {mean_recall:.3f} | MRR: {mrr:.3f}')

print('Descriptor | P@k | R@k | MRR')

print(f'LBP | {mean_precision:.3f} | {mean_recall:.3f} | {mrr:.3f}')

In [ ]:
# Display query image and its top-k retrieved neighbours
def show_retrieval(query_img, query_label, top_k_imgs, top_k_labels, top_k_dists, class_names, k=10):

    fig, axes = plt.subplots(1, k + 1, figsize=(2 * (k + 1), 3))

    axes[0].imshow(query_img)

    axes[0].set_title('Query: ' + class_names[query_label])

    axes[0].axis('off')

    for j in range(k):

        axes[j + 1].imshow(top_k_imgs[j])

        axes[j + 1].set_title(class_names[top_k_labels[j]] + '\n' + str(round(top_k_dists[j], 3)))

        axes[j + 1].axis('off')

    plt.tight_layout()

    plt.show()

# Demo retrieval on a random test image
sample_idx = np.random.randint(len(test_images))

query_img = test_images[sample_idx]

query_label = test_labels[sample_idx]

query_feat = extract_lbp(query_img)

top_k_idx, top_k_dists = retrieve(query_feat, db_features_lbp, k=k)

top_k_imgs = train_images[top_k_idx]

top_k_labels = train_labels[top_k_idx]

show_retrieval(query_img, query_label, top_k_imgs, top_k_labels, top_k_dists, class_names, k=k)